# Bounded semantic-local pilot definitions

Reusable exports for Notebook03. All training uses full-state checkpoints with exact resume and one writer. Inputs: pinned Food101 train/development memberships and three fresh JEPA models. Outputs: Notebook03 artifacts and checkpoints. Expected runtime: measured budget, target two hours, absolute three-hour cap. Import tagged definitions only; importing this notebook never starts an experiment.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)


## Imports and initialization

In [ ]:
"""Notebook03 bounded semantic-attention comparison; definitions only."""
from pathlib import Path
from contextlib import contextmanager
import copy, gc, hashlib, json, math, os, random, shlex, signal, subprocess, sys, time, traceback, uuid
import numpy as np
import torch
from fmm_jepa.models import JEPA
from fmm_jepa.geometry import clear_geometry_cache
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer, parameter_counts, move_masks, write_csv
from fmm_jepa.food101_training import (training_protocol, make_optimizer, ConsumedSampler,
    validate_training_manifest, isolated_rng, pretraining_forward, _hash, _atomic_torch,
    _rng, _restore_rng, _identity, _load_state, _finite_output, _grad_norm, _flush_timing,
    _training_performance_history, _save_state as _food_save_state)
KINDS = ('vit', 'fmm', 'semantic')

def build_model(config, kind):
    model_config=copy.deepcopy(config['model'])
    if kind=='semantic':
        from fmm_jepa.semantic_attention import build_semantic_jepa
        return build_semantic_jepa(model_config)
    if kind not in KINDS: raise ValueError('Unknown architecture')
    model_config['kind']=kind
    return JEPA(model_config)

def _save_state(*args, **kwargs):
    value=_food_save_state(*args, **kwargs)
    value['resume_notebook']='notebooks/03_Semantic_Local_Attention_Pilot.ipynb'
    value['resume_cell']='launch-detached-semantic-pilot'
    atomic_json(Path(args[0])/'manifest.json',value)
    return value

def configure_environment(config):
    if os.environ.get('CUBLAS_WORKSPACE_CONFIG') != ':4096:8':
        raise RuntimeError('Set CUBLAS_WORKSPACE_CONFIG=:4096:8 before importing torch; restart the kernel')
    torch.set_num_threads(int(config.get('cpu_threads',2)))
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    if config.get('device')=='auto': config['device']='cuda:0' if torch.cuda.is_available() else 'cpu'
    if config['device']!='cuda:0':raise ValueError('This bounded pilot is registered for cuda:0 only')
    if config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('Official test evaluation is not authorized in this pilot')
    return config

def read_config(root, path='configs/semantic_local_pilot.yaml'):
    config=json.loads((Path(root)/path).read_text())
    return configure_environment(config)

def _consumer_hash(root):
    nb=json.loads((Path(root)/'notebooks/03_Semantic_Local_Attention_Pilot.ipynb').read_text())
    return _hash([''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code'])

def artifact_path(root,run_id):
    if not run_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in run_id):
        raise ValueError('run_id must use letters, digits, underscore or hyphen')
    return Path(root)/'artifacts/notebook_03'/run_id

def status(artifact,stage,state,**details):
    value={'stage':stage,'status':state,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'pilot_status.json',value)
    print(stage,state,flush=True)
    return value

def initialize_models(config,artifact):
    """Three independent models; same nonmixer scaffold, predictor and semantic qkv/projection."""
    folder=Path(artifact)/'initialization';folder.mkdir(parents=True,exist_ok=True)
    path=folder/'manifest.json'
    identity=_hash({'model':config['model'],'seed':config['seeds']['model_init'],'kinds':KINDS})
    with isolated_rng(config['seeds']['model_init']):
        models={kind:build_model(config,kind).cpu() for kind in KINDS}
    if path.exists():
        info=json.loads(path.read_text())
        if info['identity']!=identity:raise ValueError('Initialization identity differs')
        for kind,model in models.items():
            for record in info['states'][kind].values():
                if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial-state hash mismatch')
            model.load_state_dict(torch.load(info['states'][kind]['model']['path'],map_location='cpu',weights_only=True))
        return models,info
    if list(folder.glob('*.pt')):raise ValueError('Orphan initialization states; refusing overwrite')
    source=models['vit'];copied={}
    for kind in ('fmm','semantic'):
        destination=models[kind].context.state_dict();original=source.context.state_dict();copied[kind]=[]
        for name,tensor in destination.items():
            eligible=kind=='semantic' or '.mixer.' not in name
            if eligible and name in original and tensor.shape==original[name].shape:
                tensor.copy_(original[name]);copied[kind].append('context.'+name)
        models[kind].context.load_state_dict(destination)
        models[kind].predictor.load_state_dict(source.predictor.state_dict())
        copied[kind]+=['predictor.'+name for name in source.predictor.state_dict()]
        for name in copied[kind]:
            a=dict(source.named_parameters()).get(name);b=dict(models[kind].named_parameters()).get(name)
            if a is not None and b is not None and (not torch.equal(a,b) or a.data_ptr()==b.data_ptr()):
                raise AssertionError('Initialization must be equal with independent storage')
    records={};counts={}
    for kind,model in models.items():
        model.target.load_state_dict(model.context.state_dict());model.target.requires_grad_(False).eval()
        counts[kind]=parameter_counts(model);records[kind]={}
        for name,state in [('model',model.state_dict()),('teacher',model.target.state_dict())]:
            file=folder/f'{kind}_initial_{name}.pt';_atomic_torch(file,state)
            records[kind][name]={'path':str(file.resolve()),'sha256':digest_file(file)}
    info={'identity':identity,'states':records,'parameter_counts':counts,'copied_components':copied,
          'independent_storage_verified':True,'comparison':'same scaffold and exposures; parameter/compute counts not matched',
          'snapshot_purpose':'immutable fresh initialization controls, not resumable training checkpoints'}
    atomic_json(path,info);atomic_json(Path(artifact)/'initialization_manifest.json',info)
    return models,info

@contextmanager
def deadline_guard(deadline):
    """Interrupt the main-thread worker at the persisted wall-clock cap."""
    remaining=float(deadline)-time.time()
    if remaining<=0:raise TimeoutError('Authorized pilot wall-clock budget exhausted')
    def handler(signum,frame):raise TimeoutError('Pilot wall-clock deadline reached; valid checkpoints retained')
    previous=signal.signal(signal.SIGALRM,handler)
    previous_timer=signal.setitimer(signal.ITIMER_REAL,remaining)
    try:yield
    finally:
        signal.setitimer(signal.ITIMER_REAL,*previous_timer)
        signal.signal(signal.SIGALRM,previous)


## Training

In [ ]:
def run_model(kind,config,dataset_bundle,artifact_dir,checkpoint_dir,protocol,model=None,stop_after=None,diagnostic_callback=None,deadline_unix=None):
    """Shared smoke/pilot path. stop_after bounds this invocation, never the schedule budget."""
    from fmm_jepa.food101_data import make_batch
    if kind not in KINDS:raise ValueError('Unknown model kind')
    policy=training_protocol(config);device=torch.device(config.get('device','cpu'))
    identity=_identity(config,dataset_bundle,protocol)
    artifact_dir=Path(artifact_dir);checkpoint_dir=Path(checkpoint_dir)
    artifact_dir.mkdir(parents=True,exist_ok=True);checkpoint_dir.mkdir(parents=True,exist_ok=True)
    performance_path=artifact_dir/f'{kind}_performance.json'
    previous_performance=json.loads(performance_path.read_text()) if performance_path.exists() else None
    if previous_performance is not None and previous_performance.get('identity_hash')!=_hash(identity):
        raise ValueError('Historical performance identity mismatch; use a new run ID')
    clear_geometry_cache();gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    if model is None:model=initialize_models(config,artifact_dir)[0][kind]
    model=model.to(device=device,dtype=torch.float32)
    optimizer,scheduler,group_names=make_optimizer(model,policy)
    atomic_json(artifact_dir/f'{kind}_optimizer_groups.json',group_names)
    sampler=ConsumedSampler(dataset_bundle['train_ids'],config.get('seeds',{}).get('data_order',17))
    state={'successful_updates':0,'attempted_updates':0,'image_exposures':0,'consecutive_failures':0,
           'metrics':[],'traces':[],'failures':[],'diagnostics':[]}
    start=time.perf_counter();pending=[];resumed=0;last_save=time.monotonic();manifest=None
    stop=policy['updates'] if stop_after is None else min(int(stop_after),policy['updates'])
    if stop<0:raise ValueError('stop_after cannot be negative')
    if device.type=='cuda':gc.collect();torch.cuda.empty_cache();torch.cuda.synchronize(device);torch.cuda.reset_peak_memory_stats(device)
    diagnostic_steps={0,policy['updates']//2,policy['updates']}
    def diagnostics():
        step=state['successful_updates']
        if diagnostic_callback and step in diagnostic_steps and step not in {r['step'] for r in state['diagnostics']}:
            with isolated_rng():
                record=diagnostic_callback(model,kind,step)
            state['diagnostics'].append({'step':step,'model_id':kind,'results':record})
    with run_writer(checkpoint_dir):
        if (checkpoint_dir/'manifest.json').exists():
            state,manifest=_load_state(checkpoint_dir,model,optimizer,scheduler,sampler,identity);resumed=state['successful_updates']
        elif list(checkpoint_dir.glob('*.pt')):raise ValueError('Orphan checkpoint files without manifest; refusing restart')
        diagnostics()
        if manifest is None:manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'initial')
        try:
            while state['successful_updates']<stop:
                if deadline_unix is not None and time.time() >= deadline_unix:
                    _flush_timing(state,pending,device)
                    manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'deadline_boundary')
                    break
                model.train();optimizer.zero_grad(set_to_none=True)
                state['attempted_updates']+=1;attempt=state['attempted_updates'];before_epoch=sampler.cursor//len(sampler.ids)
                values=[];trace={'model_id':kind,'attempted_update':attempt,'sample_ids':[],'occurrences':[],'augmentation_ids':[],'mask_ids':[]}
                data_seconds=0.;compute_wall=0.;visible=[];target_counts=[];micro_events=[]
                lr=float(optimizer.param_groups[0]['lr']);step=state['successful_updates']+1
                try:
                    for micro in range(policy['accumulation_steps']):
                        data_start=time.perf_counter();ids,occurrences=sampler.take(policy['microbatch'])
                        batch=make_batch(dataset_bundle['dataset'],ids,occurrences,config,training=True)
                        if batch['images'].shape[0]!=policy['microbatch']:raise ValueError('Partial microbatch refused')
                        state['image_exposures']+=len(ids)
                        # Labels intentionally never enter the JEPA call.
                        trace['sample_ids'].extend(ids);trace['occurrences'].extend(occurrences)
                        trace['augmentation_ids'].extend(batch['augmentation_ids']);trace['mask_ids'].extend(batch['mask_ids'])
                        images=batch['images'].to(device=device,dtype=torch.float32)
                        masks=move_masks(batch['masks'],device)
                        data_seconds+=time.perf_counter()-data_start
                        if device.type=='cuda':
                            event_start=torch.cuda.Event(enable_timing=True);event_end=torch.cuda.Event(enable_timing=True);event_start.record()
                        t0=time.perf_counter()
                        output=pretraining_forward(model,{'images':images,'masks':masks})
                        if not _finite_output(output):raise FloatingPointError('Nonfinite activation/loss before optimizer')
                        (output['loss']/policy['accumulation_steps']).backward()
                        values.append({k:float(v) for k,v in output['metrics'].items()})
                        visible.extend(masks['context_valid'].sum(1).detach().cpu().tolist())
                        target_counts.extend(torch.stack([v.sum(1) for v in masks['target_valid']],1).detach().cpu().flatten().tolist())
                        if device.type=='cuda':event_end.record();micro_events.append((event_start,event_end))
                        compute_wall+=time.perf_counter()-t0
                        del output,images,masks,batch
                    update_start=time.perf_counter()
                    if device.type=='cuda':update_event_start=torch.cuda.Event(enable_timing=True);update_event_end=torch.cuda.Event(enable_timing=True);update_event_start.record()
                    grads=[p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
                    if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):raise FloatingPointError('Nonfinite/absent gradients before optimizer')
                    if any(p.grad is not None for p in model.target.parameters()):raise AssertionError('Teacher gradient leakage')
                except FloatingPointError as exc:
                    optimizer.zero_grad(set_to_none=True);state['consecutive_failures']+=1
                    trace['successful']=False;trace['failure']=str(exc);state['traces'].append(trace)
                    state['failures'].append({'attempt':attempt,'reason':str(exc),'consumed_cursor':sampler.cursor})
                    if state['consecutive_failures']>=policy['max_consecutive_failures']:raise RuntimeError('Repeated nonfinite attempts exceeded cap') from exc
                    continue
                grad_norm=float(torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],policy['gradient_clip'],error_if_nonfinite=True))
                # Branch norms are measured after clipping; global norm above is before clipping.
                near_grad=_grad_norm(model.context.named_parameters(),lambda n:'.near_' in n)
                far_grad=_grad_norm(model.context.named_parameters(),lambda n:'.far_' in n or '.translation' in n)
                optimizer.step()
                if not all(bool(torch.isfinite(p).all()) for p in model.parameters()):raise RuntimeError('Nonfinite updated state; last valid checkpoint retained, offending state not published')
                momentum=scheduler.ema_for(step);model.update_teacher(momentum);scheduler.step()
                if device.type=='cuda':update_event_end.record();micro_events.append((update_event_start,update_event_end))
                compute_wall+=time.perf_counter()-update_start
                state['successful_updates']=step;state['consecutive_failures']=0
                row={'model_id':kind,'seed':int(config.get('seeds',{}).get('model_init',20260923)),
                  'phase':'pretraining','split':'train','n_grid':(int(config.get('image_size',128))//int(config['model']['patch_size']))**2,'successful_updates':step,'attempted_updates':attempt,
                  'image_exposures':state['image_exposures'],'consumed_cursor':sampler.cursor,
                  'loss':float(np.mean([x['loss'] for x in values])),'lr':lr,'ema_momentum':momentum,
                  'scheduler_position':scheduler.position,'gradient_norm_preclip':grad_norm,'clipped':grad_norm>policy['gradient_clip'],
                  'near_gradient_norm_postclip':near_grad,'far_gradient_norm_postclip':far_grad,
                  'visible_tokens_mean':float(np.mean(visible)),'visible_tokens_min':min(visible),'visible_tokens_max':max(visible),
                  'target_tokens_mean':float(np.mean(target_counts)),'effective_batch':policy['effective_batch'],
                  'microbatch':policy['microbatch'],'accumulation_steps':policy['accumulation_steps'],'data_seconds':data_seconds,
                  'teacher_gradients_absent':True,'teacher_eval':not model.target.training,
                  'failed_attempts_total':len(state['failures'])}
                for name in values[0]:
                    if name!='loss':row[name]=float(np.mean([x[name] for x in values]))
                if device.type=='cuda':
                    # Aggregate event pairs only at checkpoint boundaries to avoid synchronizing each normal update.
                    row['_cuda_event_count']=len(micro_events)
                    for ev0,ev1 in micro_events:pending.append((row,ev0,ev1))
                else:row['compute_seconds']=compute_wall
                state['metrics'].append(row);trace['successful']=True;trace['successful_update']=step;state['traces'].append(trace)
                diagnostics()
                epoch_boundary=sampler.cursor//len(sampler.ids)!=before_epoch
                if step%policy['checkpoint_every']==0 or step==stop or epoch_boundary or time.monotonic()-last_save>=900:
                    _flush_timing(state,pending,device)
                    manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'update_boundary')
                    last_save=time.monotonic()
        except BaseException as exc:
            # Save only a finite completed optimizer boundary; an in-flight partial update is recovered from the previous checkpoint.
            _flush_timing(state,pending,device)
            atomic_json(artifact_dir/f'{kind}_interruption.json',{'exception':repr(exc),'successful_updates':state['successful_updates'],
               'attempted_updates':state['attempted_updates'],'image_exposures':state['image_exposures'],
               'resume_from':str(checkpoint_dir/'manifest.json'),'partial_attempt_replayed_from_last_valid_boundary':True})
            model.cpu();optimizer.zero_grad(set_to_none=True);clear_geometry_cache();gc.collect()
            if device.type=='cuda':torch.cuda.empty_cache()
            raise
        _flush_timing(state,pending,device)
        manifest=_save_state(checkpoint_dir,state,model,optimizer,scheduler,sampler,config,protocol,identity,kind,'invocation_end')
    peak=torch.cuda.max_memory_allocated(device) if device.type=='cuda' else 0
    wall=time.perf_counter()-start
    performance=_training_performance_history(previous_performance,state,kind,device,peak,wall,resumed,identity)
    if state['metrics']:write_csv(artifact_dir/f'{kind}_training_metrics.csv',state['metrics'])
    atomic_json(artifact_dir/f'{kind}_training_traces.json',state['traces'])
    atomic_json(artifact_dir/f'{kind}_training_diagnostics.json',state['diagnostics'])
    atomic_json(artifact_dir/f'{kind}_performance.json',performance)
    model=model.cpu();optimizer.zero_grad(set_to_none=True);del optimizer,scheduler;clear_geometry_cache();gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    return {'model':model,'metrics':state['metrics'],'traces':state['traces'],'diagnostics':state['diagnostics'],
      'performance':performance,'checkpoint_manifest':manifest,'checkpoint_dir':str(checkpoint_dir),
      'resume':{'resumed_from_step':resumed,'final_step':state['successful_updates'],'configured_updates':policy['updates'],
         'complete':state['successful_updates']==policy['updates'],'actual_image_exposures':state['image_exposures']}}

## Data and protocol

In [ ]:
def reuse_food_bundle(root, config, artifact=None, verify_images=True):
    """Reuse the pinned Notebook02 split; never decode or evaluate lockbox test images."""
    from fmm_jepa.food101_data import Food101Adapter, _read_sealed, canonical_hash
    root=Path(root);prior=root/'artifacts/notebook_02'/config['source_food_run_id']
    required=['dataset_manifest.json','split_manifest.json','duplicate_audit.json']
    docs={name:json.loads((prior/name).read_text()) for name in required}
    split=docs['split_manifest.json'];audit=docs['duplicate_audit.json']
    if canonical_hash({k:v for k,v in split.items() if k!='split_sha256'})!=split['split_sha256']:
        raise ValueError('Prior split manifest checksum differs')
    if canonical_hash({k:v for k,v in audit.items() if k!='audit_sha256'})!=audit['audit_sha256']:
        raise ValueError('Prior audit checksum differs')
    cache=root/'data/food101';source=_read_sealed(cache/'dataset_manifest.json')
    if source['dataset_fingerprint']!=split['dataset_fingerprint'] or audit['audit_sha256']!=split['duplicate_audit_sha256']:
        raise ValueError('Prior split/source/audit identities differ')
    expected={'training':config['data']['train_per_class'],'development':config['data']['dev_per_class']}
    if any(split['selected_per_class'][key]!=count for key,count in expected.items()):
        raise ValueError('This pilot must preserve the audited training/development memberships')
    if int(split['split_seed'])!=int(config['data']['split_seed']):raise ValueError('Split seed differs')
    dataset=Food101Adapter(cache,source)
    train=split['selected_ids']['training'];dev=split['selected_ids']['development'];heldout=split['selected_ids']['heldout']
    if set(train)&set(dev) or (set(train)|set(dev))&set(heldout):raise ValueError('Data membership leakage')
    selected=set(train)|set(dev);verified=set()
    if verify_images:
        for entry in audit['original_record_shards']:
            shard_path=cache/'audit_shards'/entry['file']
            if digest_file(shard_path)!=entry['sha256']:raise ValueError('Original image audit shard changed')
            for record in json.loads(shard_path.read_text())['records']:
                sid=record['sample_id']
                if sid in selected:
                    image=dataset.path_for(sid)
                    if image.stat().st_size!=record['encoded_bytes'] or digest_file(image)!=record['encoded_sha256']:
                        raise ValueError('Selected train/development original image changed: '+sid)
                    verified.add(sid)
        if verified!=selected:raise ValueError('Selected images missing from original audit')
    bundle={'dataset':dataset,'dataset_manifest':docs['dataset_manifest.json'],'manifest':docs['dataset_manifest.json'],
            'split_manifest':split,'audit':audit,'fingerprint':dataset.fingerprint,'train_ids':list(train),
            'dev_ids':list(dev),'heldout_ids':list(heldout),'diagnostic_ids':list(split['diagnostic_ids'])}
    if artifact:
        for name,value in docs.items():atomic_json(Path(artifact)/name,value)
        atomic_json(Path(artifact)/'data_reuse_evidence.json',{'source_run':config['source_food_run_id'],
            'source_files':{name:digest_file(prior/name) for name in required},'verified_selected_images':len(verified),
            'test_pixels_read':0,'test_prediction_status':'NOT_RUN','train_count':len(train),'development_count':len(dev)})
    return bundle

def _protocol(root,artifact,config,bundle,initialization):
    from fmm_jepa.food101_workflow import stable_environment
    return {'schema':'fmm-jepa-semantic-pilot/v1','run_id':Path(artifact).name,'config':config,
        'source_hashes':definition_hashes(root),'consumer_notebook_sha256':_consumer_hash(root),
        'dataset_fingerprint':bundle['dataset'].fingerprint,'split_sha256':bundle['split_manifest']['split_sha256'],
        'initialization':initialization,'training':training_protocol(config),'environment':stable_environment({**config, 'effective_batch_size': int(config['training']['microbatch']) * int(config['training']['accumulation_steps'])}),
        'architectures':list(KINDS),'selection':'final common successful-update budget; no best-pretraining-loss selection',
        'comparison':'same scaffold, data, augmentation, masks, successful updates and probe budget; not parameter or compute matched',
        'primary_contrast':'semantic trained minus semantic initialization development top1',
        'secondary_contrasts':['semantic trained minus vit trained','semantic trained minus fmm trained'],
        'uncertainty':'One paired training seed. Image bootstrap omits training-seed and development-selection uncertainty.',
        'scope':'Short promise test; neither a definitive architecture ranking nor a scaling law.',
        'heldout_status':'NOT_RUN'}

def check_registered_sources(root,artifact):
    artifact=Path(artifact);protocol=json.loads((artifact/'protocol.json').read_text())
    if protocol['source_hashes']!=definition_hashes(root) or protocol['consumer_notebook_sha256']!=_consumer_hash(root):
        raise ValueError('Notebook definitions changed after registration; exact resume refused')
    if protocol['config']!=json.loads((artifact/'resolved_config.json').read_text()):raise ValueError('Registered config changed')
    return protocol


## Calibration and resume

In [ ]:
def verify_exact_resume(root, artifact, config, bundle, protocol):
    """Actual config: uninterrupted four updates equals checkpoint-restored two plus two."""
    from fmm_jepa.food101_benchmark import _food_exact_tree
    folder=Path(artifact)/'preflight';folder.mkdir(exist_ok=True)
    certificate_path=Path(artifact)/'preflight_certificate.json'
    binding={'config_hash':_hash(config),'protocol_hash':_hash(protocol)}
    if certificate_path.exists():
        certificate=json.loads(certificate_path.read_text())
        if certificate['binding']!=binding or certificate['status']!='PASS':raise ValueError('Preflight binding differs')
        for record in certificate['models'].values():
            for prefix in ('reference','resumed'):
                directory=Path(record[prefix+'_directory']);manifest=validate_training_manifest(directory)
                if manifest['successful_updates']!=4 or digest_file(directory/manifest['latest'])!=record[prefix+'_latest_sha256']:
                    raise ValueError('Preflight checkpoint changed')
        return certificate
    models,_=initialize_models(config,artifact)
    rng_path=folder/'trajectory_rng.pt';rng_manifest=folder/'trajectory_rng.json'
    if rng_manifest.exists():
        if digest_file(rng_path)!=json.loads(rng_manifest.read_text())['sha256']:raise ValueError('Preflight RNG changed')
        trajectory_rng=torch.load(rng_path,map_location='cpu',weights_only=False)
    else:
        if rng_path.exists():raise ValueError('Orphan preflight RNG snapshot')
        trajectory_rng=_rng();_atomic_torch(rng_path,trajectory_rng)
        atomic_json(rng_manifest,{'sha256':digest_file(rng_path)})
    original_rng=_rng();records={} 
    fields=['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler','streams',
            'microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields=['successful_updates','attempted_updates','image_exposures','consecutive_failures','traces','failures','diagnostics']
    timing={'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    try:
        for kind in KINDS:
            ref=Path(root)/'checkpoints'/Path(artifact).name/'preflight'/(kind+'-reference')
            split=ref.with_name(kind+'-split')
            _restore_rng(copy.deepcopy(trajectory_rng))
            reference=run_model(kind,config,bundle,folder/(kind+'-reference'),ref,protocol,model=copy.deepcopy(models[kind]),stop_after=4)
            _restore_rng(copy.deepcopy(trajectory_rng))
            partial=run_model(kind,config,bundle,folder/(kind+'-split'),split,protocol,model=copy.deepcopy(models[kind]),stop_after=2)
            marker=validate_training_manifest(split)
            if partial['resume']['final_step'] not in (2,4):raise AssertionError('Unexpected split step')
            del partial
            resumed=run_model(kind,config,bundle,folder/(kind+'-split'),split,protocol,model=copy.deepcopy(models[kind]),stop_after=4)
            left_manifest=validate_training_manifest(ref);right_manifest=validate_training_manifest(split)
            left=torch.load(ref/left_manifest['latest'],map_location='cpu',weights_only=False)
            right=torch.load(split/right_manifest['latest'],map_location='cpu',weights_only=False)
            for key in fields:_food_exact_tree(left[key],right[key],key)
            for key in state_fields:_food_exact_tree(left['state'][key],right['state'][key],'state.'+key)
            lm=[{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
            rm=[{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
            _food_exact_tree(lm,rm,'deterministic_metrics')
            if len(right_manifest['checkpoints'])<2 or resumed['resume']['resumed_from_step'] not in (2,4):
                raise AssertionError('Latest-two retention or reconstruction missing')
            if left['state']['successful_updates']!=4 or left['state']['attempted_updates']!=4:raise AssertionError('Invalid preflight update count')
            records[kind]={'status':'PASS','bitwise_equal':True,'reference_directory':str(ref.resolve()),
                'resumed_directory':str(split.resolve()),'reference_latest_sha256':digest_file(ref/left_manifest['latest']),
                'resumed_latest_sha256':digest_file(split/right_manifest['latest']),
                'full_state_fields':fields,'state_fields':state_fields,'timing_fields_excluded':sorted(timing),
                'microbatch':config['training']['microbatch'],'accumulation_steps':config['training']['accumulation_steps'],
                'schedule_budget':config['training']['updates'],'image_exposures':left['state']['image_exposures']}
            atomic_json(folder/(kind+'_certificate.json'),records[kind])
            del reference,resumed,left,right
            gc.collect();torch.cuda.empty_cache()
    finally:_restore_rng(original_rng)
    certificate={'status':'PASS','binding':binding,'models':records,'scope':'Same config, device and software; full training state equality except elapsed timing.'}
    atomic_json(certificate_path,certificate)
    return certificate

def calibrate_budget(root, artifact, config, bundle, deadline):
    """Real-image calibration chooses one equal budget before production models train."""
    folder=Path(artifact)/'calibration';folder.mkdir(exist_ok=True)
    path=folder/'calibration.json'
    if path.exists():
        value=json.loads(path.read_text())
        if value['requested_config_hash']!=_hash(config):raise ValueError('Calibration request changed')
        return value
    models,_=initialize_models(config,folder)
    protocol={'source_hashes':definition_hashes(root),'purpose':'budget calibration only; weights discarded'}
    records={}
    for kind in KINDS:
        if time.time()>deadline-1200:raise TimeoutError('Insufficient time for calibration and evaluation')
        result=run_model(kind,config,bundle,folder,Path(root)/'checkpoints'/Path(artifact).name/'calibration'/kind,
                         protocol,model=models[kind],stop_after=4,deadline_unix=deadline-1200)
        if result['resume']['final_step']!=4:raise TimeoutError('Calibration stopped before four updates')
        seconds=[row['step_seconds'] for row in result['metrics'][-2:]]
        records[kind]={'median_step_seconds':float(np.median(seconds)),'max_step_seconds':max(seconds),
                       'performance':result['performance']}
        del result
    pessimistic=sum(record['max_step_seconds'] for record in records.values())*float(config['budget']['calibration_margin'])
    target_end=min(deadline,time.time()+float(config['budget']['target_seconds']))
    available=target_end-time.time()-float(config['budget']['evaluation_reserve_seconds'])-300
    chunk=int(config['budget']['round_updates'])
    proposed=min(int(config['training']['updates']),int(available/max(pessimistic,1e-9)))
    updates=proposed//chunk*chunk
    if updates<chunk:raise RuntimeError('Measured cost leaves no useful equal-update comparison within the time cap')
    value={'requested_config_hash':_hash(config),'models':records,'selected_updates':updates,
           'pessimistic_seconds_per_three_arm_update':pessimistic,'deadline_unix':deadline,
           'estimated_training_seconds':updates*pessimistic,'discard_calibration_weights':True,
           'budget_selection_uses_only_runtime':'No representation quality inspected before fixing updates.'}
    atomic_json(path,value)
    return value


## Bounded workflow

In [ ]:
def prepare_semantic_pilot(root,run_id,config=None):
    """Prepare/calibrate/freeze/preflight. Does not launch the detached worker."""
    root=Path(root).resolve();requested=read_config(root) if config is None else configure_environment(copy.deepcopy(config))
    artifact=artifact_path(root,run_id);artifact.mkdir(parents=True,exist_ok=True)
    saved_status=artifact/'pilot_status.json'
    completed=saved_status.exists() and json.loads(saved_status.read_text()).get('status')=='COMPLETED'
    if session_exists('fmm-semantic-'+run_id) or completed:
        if json.loads((artifact/'requested_config.json').read_text())!=requested:raise ValueError('RUN_ID request changed')
        protocol=check_registered_sources(root,artifact)
        return {'artifact':str(artifact),'config':protocol['config'],'status':'COMPLETED' if completed else 'EXISTING_SESSION'}
    with run_writer(artifact):
        request_path=artifact/'requested_config.json'
        if request_path.exists() and json.loads(request_path.read_text())!=requested:raise ValueError('RUN_ID request changed')
        atomic_json(request_path,requested)
        resolved=artifact/'resolved_config.json'
        if resolved.exists():
            frozen=json.loads(resolved.read_text());check_registered_sources(root,artifact)
            if (artifact/'preflight_certificate.json').exists():
                bundle=reuse_food_bundle(root,frozen,verify_images=False)
                protocol=check_registered_sources(root,artifact)
                verify_exact_resume(root,artifact,frozen,bundle,protocol)
                return {'artifact':str(artifact),'config':frozen,'status':'READY_OR_RESUME'}
            config=frozen
        else:config=copy.deepcopy(requested)
        bundle=reuse_food_bundle(root,config,artifact,verify_images=True)
        clock_path=artifact/'budget_clock.json'
        if clock_path.exists():clock=json.loads(clock_path.read_text())
        else:
            now=time.time();clock={'started_unix':now,'deadline_unix':now+min(10800,float(config['budget']['max_seconds'])),
                'scope':'Calibration, compute benchmark, preflight, training and evaluation. Wall clock persists across restarts.'}
            atomic_json(clock_path,clock)
        deadline=clock['deadline_unix']
        if time.time()>=deadline:raise TimeoutError('This run has exhausted its authorized wall-clock budget; existing evidence retained')
        with deadline_guard(deadline):
            if not resolved.exists():
                status(artifact,'calibration','RUNNING')
                calibration=calibrate_budget(root,artifact,config,bundle,deadline)
                config['training']['updates']=calibration['selected_updates']
                config['evaluation']['diagnostic_steps']=[0,config['training']['updates']]
                models,initialization=initialize_models(config,artifact);del models
                protocol=_protocol(root,artifact,config,bundle,initialization)
                atomic_json(resolved,config);atomic_json(artifact/'protocol.json',protocol)
                atomic_json(artifact/'parameter_counts.json',initialization['parameter_counts'])
            else:protocol=check_registered_sources(root,artifact)
            from fmm_jepa.semantic_evaluation import benchmark_semantic_pilot
            status(artifact,'compute_benchmark','RUNNING')
            benchmark_semantic_pilot(root,config,bundle,artifact,deadline=deadline-900)
            status(artifact,'exact_resume_preflight','RUNNING')
            verify_exact_resume(root,artifact,config,bundle,protocol)
            status(artifact,'ready_to_launch','READY',planned_updates=config['training']['updates'],deadline_unix=deadline)
    return {'artifact':str(artifact),'config':config,'status':'READY'}

def train_three(root,artifact,run_id,config,bundle,protocol,deadline):
    models,_=initialize_models(config,artifact);trained={};chunk=int(config['budget']['round_updates'])
    budget=int(config['training']['updates']);training_deadline=deadline-float(config['budget']['evaluation_reserve_seconds'])
    calibration=json.loads((Path(artifact)/'calibration/calibration.json').read_text())
    for endpoint in range(chunk,budget+chunk,chunk):
        endpoint=min(endpoint,budget)
        completed={kind:validate_training_manifest(Path(root)/'checkpoints'/run_id/kind)['successful_updates']
                   if (Path(root)/'checkpoints'/run_id/kind/'manifest.json').exists() else 0 for kind in KINDS}
        if min(completed.values())>=endpoint:continue
        estimated=calibration['pessimistic_seconds_per_three_arm_update']*(endpoint-min(completed.values()))
        if time.time()+estimated>training_deadline:
            print('Preserving evaluation time; ending at last completed equal round.',flush=True);break
        for kind in KINDS:
            status(artifact,'training','RUNNING',model=kind,round_endpoint=endpoint,planned_updates=budget)
            result=run_model(kind,config,bundle,artifact,Path(root)/'checkpoints'/run_id/kind,protocol,
                             model=models[kind],stop_after=endpoint,deadline_unix=training_deadline)
            models[kind]=result['model'];trained[kind]=result
            if result['resume']['final_step']!=endpoint:break
        if any(kind not in trained or trained[kind]['resume']['final_step']!=endpoint for kind in KINDS):break
    for kind in KINDS:
        if kind not in trained:
            directory=Path(root)/'checkpoints'/run_id/kind
            if not (directory/'manifest.json').exists():continue
            manifest=validate_training_manifest(directory)
            trained[kind]=run_model(kind,config,bundle,artifact,directory,protocol,model=models[kind],stop_after=manifest['successful_updates'])
    steps={kind:value['resume']['final_step'] for kind,value in trained.items()}
    equal=len(steps)==len(KINDS) and len(set(steps.values()))==1
    fields=('sample_ids','occurrences','augmentation_ids','mask_ids')
    traces={kind:[{key:row[key] for key in fields} for row in value['traces']] for kind,value in trained.items()}
    paired=equal and all(traces[kind]==traces['vit'] for kind in KINDS)
    complete=paired and next(iter(steps.values()))==budget
    summary={'status':'COMPLETE' if complete else 'BOUNDED_EARLY_STOP' if paired else 'INCOMPLETE',
        'planned_updates':budget,'steps':steps,'common_successful_updates':min(steps.values()) if len(steps)==3 else 0,
        'equal_updates':equal,'paired_trace_equal':paired,'results':{kind:{**value['resume'],'performance':value['performance']} for kind,value in trained.items()},
        'checkpoint_selection':'latest equal complete-round checkpoints; no quality selection',
        'wall_clock_bounded':True}
    atomic_json(Path(artifact)/'training_summary.json',summary)
    if not paired:raise RuntimeError('Training ended without a complete equal round; no architecture quality conclusion is allowed')
    return summary

def run_semantic_pilot(root,run_id):
    if not os.environ.get('TMUX'):raise RuntimeError('Long training must run detached in tmux')
    root=Path(root).resolve();artifact=artifact_path(root,run_id)
    config=json.loads((artifact/'resolved_config.json').read_text());configure_environment(config)
    clock=json.loads((artifact/'budget_clock.json').read_text());deadline=clock['deadline_unix']
    if time.time()>=deadline:raise TimeoutError('Authorized pilot wall-clock budget exhausted')
    def deadline_signal(signum,frame):raise TimeoutError('Hard pilot wall-clock deadline reached; valid checkpoints retained')
    previous=signal.signal(signal.SIGALRM,deadline_signal)
    signal.setitimer(signal.ITIMER_REAL,max(1,deadline-time.time()))
    try:
        with run_writer(artifact):
            try:
                protocol=check_registered_sources(root,artifact)
                bundle=reuse_food_bundle(root,config,verify_images=False)
                verify_exact_resume(root,artifact,config,bundle,protocol)
                training=train_three(root,artifact,run_id,config,bundle,protocol,deadline)
                status(artifact,'frozen_feature_evaluation','RUNNING',training=training)
                from fmm_jepa.semantic_evaluation import evaluate_semantic_pilot
                result=evaluate_semantic_pilot(root,config,bundle,run_id,artifact,deadline=deadline-15)
                evaluation_status=result.get('status','UNKNOWN') if isinstance(result,dict) else 'UNKNOWN'
                if evaluation_status in ('INCOMPLETE','FAILED','TIMED_OUT'):
                    status(artifact,'evaluation_cutoff','INCOMPLETE',elapsed_seconds=time.time()-clock['started_unix'],evaluation_status=evaluation_status)
                    return result
                status(artifact,'complete','COMPLETED',elapsed_seconds=time.time()-clock['started_unix'],training_status=training['status'])
                return result
            except BaseException:
                status(artifact,'interrupted','FAILED',traceback=traceback.format_exc(),
                       resume='Same RUN_ID and unchanged sources/config, Run All; deadline never resets automatically.')
                raise
    finally:
        signal.setitimer(signal.ITIMER_REAL,0);signal.signal(signal.SIGALRM,previous)

def session_exists(session):
    return subprocess.run(['tmux','has-session','-t','='+session],capture_output=True).returncode==0

def launch_semantic_pilot(root,run_id):
    import nbformat,shutil
    root=Path(root).resolve();artifact=artifact_path(root,run_id);session='fmm-semantic-'+run_id
    if session_exists(session):return {'status':'EXISTING_SESSION','session':session,'attach':'tmux attach -t '+session}
    status_path=artifact/'pilot_status.json'
    if status_path.exists() and json.loads(status_path.read_text()).get('status')=='COMPLETED':
        return {'status':'COMPLETED','report':str(artifact/'semantic_results.md')}
    if not shutil.which('tmux'):raise RuntimeError('tmux is required')
    with run_writer(artifact):
        protocol=check_registered_sources(root,artifact);config=protocol['config']
        certificate=json.loads((artifact/'preflight_certificate.json').read_text())
        if certificate['status']!='PASS' or certificate['binding']!={'config_hash':_hash(config),'protocol_hash':_hash(protocol)}:
            raise RuntimeError('Exact-config preflight required before launch')
        clock=json.loads((artifact/'budget_clock.json').read_text())
        if time.time()>=clock['deadline_unix']:raise TimeoutError('Pilot deadline exhausted; no automatic extension')
        main=nbformat.read(root/'notebooks/03_Semantic_Local_Attention_Pilot.ipynb',as_version=4)
        setup=next(c.source for c in main.cells if c.get('id')=='bootstrap-semantic-pilot')
        code=setup+'\nfrom fmm_jepa.semantic_pilot import run_semantic_pilot\nrun_semantic_pilot(ROOT, '+repr(run_id)+')\n'
        worker=artifact/'semantic_worker.ipynb'
        if worker.exists():
            saved=nbformat.read(worker,as_version=4)
            if [c.source for c in saved.cells if c.cell_type=='code']!=[code]:raise RuntimeError('Worker code differs')
        else:
            nb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Detached semantic-local pilot worker\nFull state under checkpoints/'+run_id+'. Resume from Notebook03 with same RUN_ID.'),nbformat.v4.new_code_cell(code)],
                 metadata={'kernelspec':{'display_name':'Python (ahr-go)','language':'python','name':'ahr-go'}})
            nbformat.write(nb,worker)
        command=[sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.timeout=-1',
                 '--ExecutePreprocessor.kernel_name=ahr-go',str(worker)]
        launch={'session':session,'attach':'tmux attach -t '+session,'detach':'Ctrl-b, then d','worker':str(worker),
                'checkpoint_root':str(root/'checkpoints'/run_id),'deadline_unix':clock['deadline_unix'],'command':command}
        atomic_json(artifact/'launch.json',launch);status(artifact,'launch','QUEUED')
        shell='exec '+shlex.join(command)+' > '+shlex.quote(str(artifact/'tmux_driver.log'))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(root),'bash','-c',shell],check=True)
    return {'status':'LAUNCHED',**launch}

def wait_for_semantic_pilot(root,run_id,poll_seconds=10):
    from IPython.display import display,Markdown
    if not 1<=poll_seconds<=60:raise ValueError('Poll between 1 and 60 seconds')
    artifact=artifact_path(root,run_id);session='fmm-semantic-'+run_id
    handle=display(Markdown('Connecting to detached pilot…'),display_id=True);missing=None
    while True:
        path=artifact/'pilot_status.json';value=json.loads(path.read_text()) if path.exists() else {'status':'QUEUED'}
        lines=['**Semantic-local pilot: '+value['status']+'**','',str(value.get('stage','starting'))]
        for kind in KINDS:
            file=Path(root)/'checkpoints'/run_id/kind/'manifest.json'
            if file.exists():
                m=json.loads(file.read_text());lines.append(f"- {kind}: {m['successful_updates']} saved updates; {m['image_exposures']} images.")
        lines+=['','Training survives notebook disconnects. Attach: `tmux attach -t '+session+'`.']
        handle.update(Markdown('\n'.join(lines)))
        if value['status']=='COMPLETED':return value
        if value['status'] in ('FAILED','CANCELLED','TIMED_OUT','INCOMPLETE'):raise RuntimeError(value.get('traceback',str(value)))
        if session_exists(session):missing=None
        else:
            missing=time.monotonic() if missing is None else missing
            if time.monotonic()-missing>30:raise RuntimeError('Worker exited without completion; inspect '+str(artifact/'tmux_driver.log'))
        time.sleep(poll_seconds)
